# WHAT CHANGED AND WHY

**Previous run problems:**
  Train F1 = 0.97, Val F1 = 0.38 -> 0.59 gap
  Val loss exploding from epoch 1
  Model memorizing 1994-2018, failing on 2020-2024

**Root cause:**
  128 hidden dim -> ~800k parameters for 5000 rows
  That is 160 parameters per training row
  Model has enough capacity to memorize everything

**Fix: reduce capacity + heavy regularization**

  **hidden_dim:   128 -> 64**
    800k params -> 200k params
    40 params per row instead of 160
    Cannot memorize — forced to generalize

  **lstm_layers:  2 -> 1**
    Two LSTM layers on 60-day windows is overkill
    One layer captures temporal patterns fine

  **dropout:      0.3 -> 0.5**
    Train F1=0.97 means model is near-perfectly
    memorizing sequences
    0.5 zeros half the neurons each forward pass
    Forces redundant representations

  **label_smoothing: 0.0 -> 0.1**
    Stops model becoming 99% confident on train
    Soft targets prevent loss going to zero on train
    while val loss explodes
    Add to F.cross_entropy() call: label_smoothing=0.1

  **lr:           3e-4 -> 1e-4**
    Slower learning gives regularization more time
    to work before model memorizes training set

  **weight_decay: 1e-3 -> 5e-3**
    5x stronger L2 penalty on weights
    Penalizes large weights that enable memorization

  **batch_size:   64 -> 128**
    Larger batches = smoother gradient estimates
    Less chance of fitting to individual sequences

  **grad_clip:    1.0 -> 0.5**
    Tighter gradient clipping

# CrashSignal — Retraining v3 (Overfitting Fix)

**Problem in v2:** Train F1=0.97, Val F1=0.38 — pure overfit
**Fix:** Reduce model capacity + aggressive regularization

| Parameter    | v2    | v3    | Why                        |
|-------------|-------|-------|----------------------------|
| hidden_dim  | 128   | 64    | 800k->200k params           |
| lstm_layers | 2     | 1     | simpler temporal model     |
| dropout     | 0.3   | 0.5   | force redundant features   |
| label_smooth| 0.0   | 0.1   | prevent overconfidence     |
| lr          | 3e-4  | 1e-4  | slower memorization        |
| weight_decay| 1e-3  | 5e-3  | stronger L2 regularization |
| batch_size  | 64    | 128   | smoother gradients         |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import os, json, joblib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    f1_score, roc_auc_score,
    classification_report, confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm

warnings.filterwarnings("ignore")
plt.style.use("dark_background")

torch.manual_seed(42)
np.random.seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU. Enable in Settings -> Accelerator")

In [ ]:
# ── Data ──────────────────────────────────────────
TRAIN_END    = "2018-12-31"
VAL_START    = "2020-01-01"
WINDOW_SIZE  = 60
CLIP_LOW     = 0.02
CLIP_HIGH    = 0.98

# ── Model — REDUCED CAPACITY ──────────────────────
HIDDEN_DIM   = 64       # was 128
LSTM_LAYERS  = 1        # was 2
NUM_HEADS    = 4
NUM_CLASSES  = 3
DROPOUT      = 0.5      # was 0.3

# ── Loss ──────────────────────────────────────────
ALPHA        = 0.5      # CE weight
BETA         = 0.3      # ordinal weight
GAMMA        = 0.2      # smoothness weight
LABEL_SMOOTH = 0.1      # was 0.0

# ── Training — STRONGER REGULARIZATION ────────────
LR           = 1e-4     # was 3e-4
WEIGHT_DECAY = 5e-3     # was 1e-3
BATCH_SIZE   = 128      # was 64
NUM_EPOCHS   = 80
PATIENCE     = 20
GRAD_CLIP    = 0.5      # was 1.0
COSINE_TMAX  = 80
COSINE_EMIN  = 1e-5

# ── Paths ─────────────────────────────────────────
DATA_PATH    = "master_df.parquet"
FEAT_PATH    = "feature_columns.json"
CRISIS_PATH  = "crisis_periods.json"
OUT_DIR      = "/kaggle/working"

print("Hyperparameters loaded.")
print(f"hidden_dim={HIDDEN_DIM}  lstm_layers={LSTM_LAYERS}")
print(f"dropout={DROPOUT}  label_smooth={LABEL_SMOOTH}")
print(f"lr={LR}  wd={WEIGHT_DECAY}  bs={BATCH_SIZE}")

In [ ]:
for path in [DATA_PATH,
             f"/kaggle/input/datasets/adityasai848/artifacts/{DATA_PATH}"]:
    if os.path.exists(path):
        df = pd.read_parquet(path)
        print(f"Loaded: {path}")
        break
else:
    raise FileNotFoundError(
        f"Cannot find {DATA_PATH}. Upload it first."
    )

for path in [FEAT_PATH,
             f"/kaggle/input/datasets/adityasai848/artifacts/{FEAT_PATH}"]:
    if os.path.exists(path):
        with open(path) as f:
            feature_cols = json.load(f)
        break
else:
    raise FileNotFoundError(f"Cannot find {FEAT_PATH}")

for path in [CRISIS_PATH,
             f"/kaggle/input/datasets/adityasai848/artifacts/{CRISIS_PATH}"]:
    if os.path.exists(path):
        with open(path) as f:
            crisis_periods = json.load(f)
        break
else:
    raise FileNotFoundError(f"Cannot find {CRISIS_PATH}")

print(f"DataFrame:  {df.shape}")
print(f"Features:   {len(feature_cols)}")
print(f"Crises:     {len(crisis_periods)}")
print(f"Date range: {df.index[0].date()} -> "
      f"{df.index[-1].date()}")

counts = df["crisis_label"].value_counts().sort_index()
total  = len(df)
for label, name in zip([0,1,2],
                       ["Normal","Stress","Crisis"]):
    n = counts.get(label, 0)
    print(f"  {name:8s}: {n:5d} ({n/total:.1%})")

In [ ]:
X_raw = df[feature_cols].copy()

# 1. Convert everything to numeric (coerce errors to NaN)
for col in X_raw.columns:
    X_raw[col] = pd.to_numeric(X_raw[col], errors='coerce')

# 2. Replace infinities with NaN
X_raw = X_raw.replace([np.inf, -np.inf], np.nan)

# 3. Fill NaNs with column median, and 0.0 as final fallback
for col in X_raw.columns:
    col_median = X_raw[col].median()
    if pd.isna(col_median):
        col_median = 0.0
    X_raw[col] = X_raw[col].fillna(col_median)
    
# 4. Aggressive final fallback for any bizarre remaining NaNs
X_raw = X_raw.fillna(0.0)

for col in X_raw.columns:
    lo = X_raw[col].quantile(CLIP_LOW)
    hi = X_raw[col].quantile(CLIP_HIGH)
    if pd.isna(lo): lo = 0.0
    if pd.isna(hi): hi = 0.0
    X_raw[col] = X_raw[col].clip(lo, hi)

y     = df["crisis_label"].values
dates = df.index

assert not X_raw.isnull().any().any(), \
    "NaN values remain after cleaning"
print(f"Features clean: {X_raw.shape}, NaN: 0")

train_mask = df.index <= TRAIN_END
val_mask   = df.index >= VAL_START

X_train, X_val   = X_raw[train_mask].values, X_raw[val_mask].values
y_train, y_val   = y[train_mask], y[val_mask]
d_train, d_val   = dates[train_mask], dates[val_mask]

print(f"Train: {len(X_train)} days "
      f"({d_train[0].date()} -> {d_train[-1].date()})")
print(f"Val:   {len(X_val)} days "
      f"({d_val[0].date()} -> {d_val[-1].date()})")

scaler         = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)

def make_sequences(X, y, w):
    Xs = np.array([X[i:i+w]
                   for i in range(len(X)-w)],
                  dtype=np.float32)
    ys = np.array([y[i+w] for i in range(len(X)-w)])
    return Xs, ys

X_tr, y_tr = make_sequences(X_train_scaled,
                             y_train, WINDOW_SIZE)
X_va, y_va = make_sequences(X_val_scaled,
                             y_val,   WINDOW_SIZE)

print(f"Train sequences: {X_tr.shape}")
print(f"Val sequences:   {X_va.shape}")

raw_w  = compute_class_weight(
    "balanced", classes=np.unique(y_tr), y=y_tr
)
cw     = torch.tensor(raw_w,
                      dtype=torch.float32).to(device)

print(f"Class weights (raw): {raw_w.round(3)}")

class StressDS(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

tr_loader = DataLoader(StressDS(X_tr, y_tr),
    batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True)
va_loader = DataLoader(StressDS(X_va, y_va),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2)

print(f"Train batches: {len(tr_loader)}")
print(f"Val batches:   {len(va_loader)}")

In [ ]:
class GRN(nn.Module):
    def __init__(self, in_d, hid_d, out_d, drop=DROPOUT):
        super().__init__()
        self.fc1  = nn.Linear(in_d, hid_d)
        self.fc2  = nn.Linear(hid_d, out_d)
        self.gate = nn.Linear(in_d, out_d)
        self.norm = nn.LayerNorm(out_d)
        self.drop = nn.Dropout(drop)
        self.elu  = nn.ELU()
        self.res  = (nn.Linear(in_d, out_d)
                     if in_d != out_d
                     else nn.Identity())

    def forward(self, x):
        r = self.res(x)
        h = self.drop(self.fc2(self.elu(self.fc1(x))))
        g = torch.sigmoid(self.gate(x))
        return self.norm(g * h + (1 - g) * r)


class VSN(nn.Module):
    def __init__(self, in_d, hid_d, drop=DROPOUT):
        super().__init__()
        self.in_d    = in_d
        self.var_nns = nn.ModuleList([
            nn.Sequential(
                nn.Linear(1, hid_d), nn.ELU(),
                nn.Dropout(drop),
                nn.Linear(hid_d, hid_d)
            ) for _ in range(in_d)
        ])
        self.wnet = nn.Sequential(
            nn.Linear(in_d, hid_d), nn.ELU(),
            nn.Linear(hid_d, in_d), nn.Softmax(dim=-1)
        )
        self.proj = nn.Linear(hid_d, hid_d)

    def forward(self, x):
        w    = self.wnet(x.mean(dim=1))
        outs = [self.var_nns[i](x[:,:,i:i+1])
                for i in range(self.in_d)]
        stk  = torch.stack(outs, dim=-1)
        sel  = (stk * w.unsqueeze(1).unsqueeze(2)
                ).sum(dim=-1)
        return self.proj(sel), w


class CrashSignalTFT(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        H = HIDDEN_DIM
        self.vsn  = VSN(input_dim, H)
        self.lstm = nn.LSTM(
            H, H, LSTM_LAYERS,
            batch_first=True,
            dropout=DROPOUT if LSTM_LAYERS > 1 else 0
        )
        self.grn1   = GRN(H, H, H)
        self.attn   = nn.MultiheadAttention(
            H, NUM_HEADS, dropout=DROPOUT,
            batch_first=True
        )
        self.grn2   = GRN(H, H, H)
        self.norm   = nn.LayerNorm(H)
        self.clf    = nn.Sequential(
            nn.Linear(H, H//2), nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(H//2, NUM_CLASSES)
        )
        self.stress = nn.Sequential(
            nn.Linear(H, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid()
        )

    def forward(self, x):
        x, w     = self.vsn(x)
        x, _     = self.lstm(x)
        x        = self.grn1(x)
        x2, _    = self.attn(x, x, x)
        x        = self.grn2(x2 + x)
        x        = self.norm(x)
        f        = x[:, -1, :]
        return self.clf(f), self.stress(f) * 100, w


model    = CrashSignalTFT(len(feature_cols)).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}  "
      f"(was ~800k, now smaller)")

dummy = torch.randn(4, WINDOW_SIZE,
                    len(feature_cols)).to(device)
lg, st, wt = model(dummy)
assert lg.shape == (4, NUM_CLASSES)
assert st.shape == (4, 1)
assert wt.shape == (4, len(feature_cols))
print("Forward pass OK [SUCCESS]")

In [ ]:
class HybridStressLoss(nn.Module):
    """
    CE + Ordinal + Smoothness.
    Key addition vs v2: label_smoothing=0.1 in CE.
    Prevents model becoming 99% confident on train
    while val loss explodes.
    """
    def __init__(self, class_weights):
        super().__init__()
        self.cw = class_weights

    def ce_loss(self, logits, targets):
        # label_smoothing=0.1 is the new addition
        return F.cross_entropy(
            logits, targets,
            weight=self.cw,
            label_smoothing=LABEL_SMOOTH
        )

    def ordinal_loss(self, logits, targets):
        probs = F.softmax(logits, dim=-1)
        loss  = 0.0
        for k in range(NUM_CLASSES - 1):
            p    = probs[:, k+1:].sum(dim=-1)
            b    = (targets > k).float()
            loss += F.binary_cross_entropy(
                p.clamp(1e-7, 1-1e-7), b
            )
        return loss / (NUM_CLASSES - 1)

    def smooth_loss(self, stress):
        if stress.shape[0] < 2:
            return torch.tensor(0.0,
                                device=stress.device)
        s    = stress.squeeze(-1)
        diff = (s[1:] - s[:-1]) / 100.0
        return diff.pow(2).mean()

    def forward(self, logits, stress, targets):
        l_ce  = self.ce_loss(logits, targets)
        l_ord = self.ordinal_loss(logits, targets)
        l_smo = self.smooth_loss(stress)
        total = (ALPHA * l_ce +
                 BETA  * l_ord +
                 GAMMA * l_smo)
        return total, {
            "ce":  round(l_ce.item(),  4),
            "ord": round(l_ord.item(), 4),
            "smo": round(l_smo.item(), 4)
        }


criterion = HybridStressLoss(cw)

dummy_lg = torch.randn(8, NUM_CLASSES).to(device)
dummy_st = torch.rand(8, 1).to(device) * 100
dummy_y  = torch.randint(0, 3, (8,)).to(device)
loss, c  = criterion(dummy_lg, dummy_st, dummy_y)
print(f"Loss test: {loss.item():.4f}  comps: {c}")
print("Hybrid loss OK [SUCCESS]")

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=COSINE_TMAX, eta_min=COSINE_EMIN
)

history   = {k: [] for k in [
    "tr_loss","va_loss","tr_f1","va_f1","va_auc"
]}
best_f1   = 0.0
pat_ctr   = 0
best_path = f"{OUT_DIR}/crashsignal_model.pth"

print("=" * 65)
print("CRASHSIGNAL v3 — OVERFITTING FIX")
print(f"  hidden={HIDDEN_DIM}  layers={LSTM_LAYERS}  "
      f"drop={DROPOUT}  smooth={LABEL_SMOOTH}")
print(f"  lr={LR}  wd={WEIGHT_DECAY}  bs={BATCH_SIZE}")
print(f"  target: train/val F1 gap < 0.20")
print("=" * 65)

for ep in range(NUM_EPOCHS):

    model.train()
    tl, tp, tlb, tc = [], [], [], {}

    for xb, yb in tr_loader:
        xb, yb      = xb.to(device), yb.to(device)
        lg, st, _   = model(xb)
        loss, comps = criterion(lg, st, yb)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model.parameters(), GRAD_CLIP
        )
        optimizer.step()
        tl.append(loss.item())
        tp.extend(lg.argmax(1).cpu().numpy())
        tlb.extend(yb.cpu().numpy())
        for k, v in comps.items():
            tc.setdefault(k, []).append(v)

    scheduler.step()
    tr_loss = np.mean(tl)
    tr_f1   = f1_score(tlb, tp,
                       average="macro",
                       zero_division=0)

    model.eval()
    vl, vp, vlb, vpr = [], [], [], []
    with torch.no_grad():
        for xb, yb in va_loader:
            xb, yb    = xb.to(device), yb.to(device)
            lg, st, _ = model(xb)
            loss, _   = criterion(lg, st, yb)
            probs     = F.softmax(lg, dim=-1)
            vl.append(loss.item())
            vp.extend(lg.argmax(1).cpu().numpy())
            vlb.extend(yb.cpu().numpy())
            vpr.extend(probs.cpu().numpy())

    va_loss = np.mean(vl)
    va_f1   = f1_score(vlb, vp,
                       average="macro",
                       zero_division=0)
    try:
        va_auc = roc_auc_score(
            vlb, vpr,
            multi_class="ovr", average="macro"
        )
    except Exception:
        va_auc = 0.0

    history["tr_loss"].append(tr_loss)
    history["va_loss"].append(va_loss)
    history["tr_f1"].append(tr_f1)
    history["va_f1"].append(va_f1)
    history["va_auc"].append(va_auc)

    gap = tr_f1 - va_f1
    print(f"Ep {ep+1:02d} | "
          f"TrL={tr_loss:.4f} VaL={va_loss:.4f} | "
          f"TrF1={tr_f1:.4f} VaF1={va_f1:.4f} | "
          f"gap={gap:.3f} AUC={va_auc:.4f}")

    if va_f1 > best_f1:
        best_f1 = va_f1
        pat_ctr = 0
        torch.save(model.state_dict(), best_path)
        print(f"  [SUCCESS] Saved F1={va_f1:.4f}")
    else:
        pat_ctr += 1
        if pat_ctr >= PATIENCE:
            print(f"Early stop at epoch {ep+1}")
            break

gap_final = history["tr_f1"][-1] - history["va_f1"][-1]
print(f"\nBest Val F1:  {best_f1:.4f}")
print(f"Best Val AUC: {max(history['va_auc']):.4f}")
print(f"Final gap:    {gap_final:.4f} "
      f"(target < 0.20)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4),
                         facecolor="#0a0e1a")
for ax in axes:
    ax.set_facecolor("#0a0e1a")
    ax.tick_params(colors="white")

ep_r = range(1, len(history["tr_loss"]) + 1)

axes[0].plot(ep_r, history["tr_loss"],
             color="#ff6b6b", label="Train")
axes[0].plot(ep_r, history["va_loss"],
             color="#4a9eff", label="Val")
axes[0].set_title("Loss — should converge",
                  color="white")
axes[0].legend(facecolor="#0a0e1a",
               labelcolor="white")

axes[1].plot(ep_r, history["tr_f1"],
             color="#ff6b6b", label="Train")
axes[1].plot(ep_r, history["va_f1"],
             color="#4a9eff", label="Val")
axes[1].axhline(0.55, color="#00ffaa",
                linestyle="--", label="Target")
axes[1].set_title("F1 — gap should close",
                  color="white")
axes[1].legend(facecolor="#0a0e1a",
               labelcolor="white")

axes[2].plot(ep_r, history["va_auc"],
             color="#ffd93d")
axes[2].axhline(0.75, color="#00ffaa",
                linestyle="--", label="Target 0.75")
axes[2].set_title("Val AUC — should stabilize",
                  color="white")
axes[2].legend(facecolor="#0a0e1a",
               labelcolor="white")

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/training_curves.png",
            dpi=120, facecolor="#0a0e1a")
plt.show()

In [ ]:
model.load_state_dict(torch.load(
    best_path, map_location=device
))
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for xb, yb in va_loader:
        xb       = xb.to(device)
        lg, _, _ = model(xb)
        probs    = F.softmax(lg, dim=-1)
        all_preds.extend(lg.argmax(1).cpu().numpy())
        all_labels.extend(yb.numpy())
        all_probs.extend(probs.cpu().numpy())

print(classification_report(
    all_labels, all_preds,
    target_names=["Normal","Stress","Crisis"],
    digits=4
))

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(7, 5),
                       facecolor="#0a0e1a")
ax.set_facecolor("#0a0e1a")
sns.heatmap(cm, annot=True, fmt="d",
    xticklabels=["Normal","Stress","Crisis"],
    yticklabels=["Normal","Stress","Crisis"],
    cmap="Reds", ax=ax)
ax.set_xlabel("Predicted", color="white")
ax.set_ylabel("Actual",    color="white")
ax.set_title("Confusion Matrix (2020-2024)",
             color="white")
ax.tick_params(colors="white")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/confusion_matrix.png",
            dpi=120, facecolor="#0a0e1a")
plt.show()

prev_run = 348
now      = cm[0][2]
print(f"\nNormal->Crisis: {now}  (v2 was {prev_run})")
print(f"Stress F1 improvement check:")
print(f"  v2 Stress F1: 0.2947")
print(f"  v3 Stress F1: "
      f"{f1_score(all_labels, all_preds, average=None, zero_division=0)[1]:.4f}")

In [ ]:
X_all = scaler.transform(X_raw.values)

def gen_scores(model, X, dates, y, w, bs=256):
    model.eval()
    scores, preds, wts = [], [], []
    seqs = np.array([X[i:i+w]
                     for i in range(len(X)-w)],
                    dtype=np.float32)
    for i in tqdm(range(0, len(seqs), bs),
                  desc="Scoring"):
        b = torch.tensor(seqs[i:i+bs]).to(device)
        with torch.no_grad():
            lg, st, wt = model(b)
        scores.extend(
            st.squeeze(-1).cpu().numpy().tolist()
        )
        preds.extend(
            lg.argmax(1).cpu().numpy().tolist()
        )
        wts.append(wt.mean(dim=0).cpu().numpy())

    n   = len(scores)
    out = pd.DataFrame({
        "date":            dates[w:w+n],
        "stress_score":    scores,
        "predicted_label": preds,
        "crisis_label":    y[w:w+n]
    }).set_index("date")
    return out, np.mean(wts, axis=0)

scores_df, avg_wt = gen_scores(
    model, X_all, dates, y, WINDOW_SIZE
)

print(f"Scores: {len(scores_df)} days")
print(scores_df["stress_score"].describe().round(2))
nan_ct = scores_df["stress_score"].isnull().sum()
print(f"NaN: {nan_ct}  {'[SUCCESS]' if nan_ct==0 else '[FAIL]'}")

In [ ]:
crisis_checks = [
    ("2008 GFC",   "2008-09-01","2009-03-09", 70),
    ("2020 COVID", "2020-02-19","2020-04-01", 65),
    ("2022 Bear",  "2022-01-01","2022-10-12", 55),
    ("2023 SVB",   "2023-03-08","2023-04-01", 55),
]

print("=" * 55)
print("CRISIS STRESS VALIDATION")
print("=" * 55)
all_pass = True
results  = {}
for name, s, e, target in crisis_checks:
    mask = ((scores_df.index >= s) &
            (scores_df.index <= e))
    if mask.any():
        peak   = scores_df.loc[mask,"stress_score"].max()
        passed = peak >= target
        all_pass = all_pass and passed
        results[name] = (peak, target, passed)
        print(f"{name:15s} peak={peak:5.1f}  "
              f"target>{target}  "
              f"{'[SUCCESS]' if passed else '[FAIL]'}")
    else:
        print(f"{name:15s} no data")
print("=" * 55)
print("ALL PASS [SUCCESS]" if all_pass else
      "SOME FAILED [FAIL] — model may not detect crises")

In [ ]:
fig, ax = plt.subplots(figsize=(18, 4),
                       facecolor="#0a0e1a")
ax.set_facecolor("#0a0e1a")
ax.plot(scores_df.index, scores_df["stress_score"],
        color="#4a9eff", lw=0.7)
for cp in crisis_periods:
    s   = pd.Timestamp(cp["start"])
    e   = pd.Timestamp(cp["end"])
    col = "#ff4444" if cp["severity"]==2 else "#ff8c00"
    ax.axvspan(s, e,
               alpha=0.25 if cp["severity"]==2 else 0.12,
               color=col)
    mid = s + (e-s)/2
    ax.text(mid, 92, cp["name"][:12],
            fontsize=5, color=col,
            ha="center", rotation=40)
for lvl, col, lbl in [
    (75,"#ff4444","High"),
    (50,"#ff8c00","Elevated")]:
    ax.axhline(lvl, color=col, lw=0.8,
               linestyle="--", alpha=0.5,
               label=lbl)
ax.set_ylim(0,100)
ax.set_title("CrashSignal — 30 Year Stress History",
             color="white", fontsize=13)
ax.set_xlabel("Year", color="white")
ax.set_ylabel("Stress Score", color="white")
ax.tick_params(colors="white")
ax.legend(facecolor="#0a0e1a",
          labelcolor="white", fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/stress_timeline.png",
            dpi=120, facecolor="#0a0e1a")
plt.show()

imp_df = pd.DataFrame({
    "indicator":  feature_cols[:len(avg_wt)],
    "importance": avg_wt
}).sort_values("importance", ascending=False)
print("TOP 10:")
print(imp_df.head(10).to_string(index=False))

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)

joblib.dump(scaler,
    f"{OUT_DIR}/crashsignal_scaler.pkl")
json.dump({
    "input_dim":    len(feature_cols),
    "hidden_dim":   HIDDEN_DIM,
    "lstm_layers":  LSTM_LAYERS,
    "num_heads":    NUM_HEADS,
    "num_classes":  NUM_CLASSES,
    "dropout":      DROPOUT,
    "window_size":  WINDOW_SIZE,
    "feature_cols": feature_cols,
    "architecture": "tft-v3",
    "loss":         "hybrid-ce-ordinal-smooth",
    "label_smooth": LABEL_SMOOTH
}, open(f"{OUT_DIR}/model_config.json","w"), indent=2)
scores_df.reset_index().to_parquet(
    f"{OUT_DIR}/historical_stress.parquet"
)
imp_df.to_csv(
    f"{OUT_DIR}/variable_importance.csv", index=False
)

from sklearn.metrics.pairwise import cosine_similarity
def find_analogs(today_vec, all_vecs,
                 all_dates, sc, k=3):
    sims    = cosine_similarity(
        today_vec.reshape(1,-1), all_vecs
    )[0]
    top_idx = sims.argsort()[-k-1:-1][::-1]
    return [{
        "date":         str(all_dates[i].date()),
        "similarity":   round(float(sims[i]),4),
        "stress_score": round(float(sc[i]),1),
        "label": ["Normal","Stress","Crisis"][int(y[i])]
    } for i in top_idx]

analogs = find_analogs(
    X_all[-1], X_all, dates,
    scores_df["stress_score"].values
)
json.dump(analogs,
    open(f"{OUT_DIR}/today_analogs.json","w"),
    indent=2)

best_auc  = max(history["va_auc"])
f1_pass   = best_f1  >= 0.55
auc_pass  = best_auc >= 0.75
gap_final = history["tr_f1"][-1] - history["va_f1"][-1]

print(f"""
============================================================
CRASHSIGNAL v3 — FINAL SUMMARY
============================================================
Best Val F1:  {best_f1:.4f}  (target>0.55)  {'[SUCCESS]' if f1_pass  else '[FAIL]'}
Best Val AUC: {best_auc:.4f}  (target>0.75)  {'[SUCCESS]' if auc_pass else '[FAIL]'}
Train/Val gap:{gap_final:.4f}  (target<0.20)  {'[SUCCESS]' if gap_final<0.20 else '[FAIL]'}

CRISIS PEAKS:""")
for name,(peak,target,passed) in results.items():
    print(f"  {name:15s} {peak:5.1f}/100  "
          f"target>{target}  {'[SUCCESS]' if passed else '[FAIL]'}")

print(f"""
ARTIFACTS:
  [SUCCESS] {OUT_DIR}/crashsignal_model.pth
  [SUCCESS] {OUT_DIR}/crashsignal_scaler.pkl
  [SUCCESS] {OUT_DIR}/model_config.json
  [SUCCESS] {OUT_DIR}/historical_stress.parquet
  [SUCCESS] {OUT_DIR}/variable_importance.csv
  [SUCCESS] {OUT_DIR}/today_analogs.json

DOWNLOAD ALL 6 FILES FROM OUTPUT TAB NOW.
============================================================
""")